# Top Movers Playground

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/portfolio-top-movers/blob/notebook/notebooks/top_movers_playground.ipynb)

A hands-on, **fully customizable** version of the Portfolio Top Movers report. Edit the **Config panel** below and re-run — no server required.

This notebook reuses the tested pipeline in [`services/`](../services) and [`config/`](../config) and layers on the improvements requested in client feedback:

1. **Time period actually matters.** The lookback drives *both* the news window *and* the price-change window used to rank movers (the app previously always ranked on a hard-coded 1-day change).
2. **Pick your topics + add your own.** Choose which named topics to include (e.g. *Products*, *Supply Chain*) and add custom ones (e.g. *Product News*).
3. **Summaries capped at the 3 most important bullets** per stock (ranked by materiality), instead of a random-length note.

### How to use

1. Run the **Setup** and **API keys** cells.
2. Edit the **Config panel** to taste.
3. Run **Build topics** -> **Run pipeline** -> **Results**.

## 1. Setup

Locates the repo (or clones it on Colab), installs dependencies, and puts the project on the import path.

In [21]:
import importlib.util
import os
import shutil
import subprocess
import sys

# Do NOT import pandas (or other runtime deps) before installing them below.

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bigdata-com/portfolio-top-movers.git"
REPO_DIR = "portfolio-top-movers"
BRANCH = "notebook"

# (import_name, pip/uv package spec)
RUNTIME_PACKAGES = [
    ("aiohttp", "aiohttp>=3.9.0"),
    ("requests", "requests>=2.31.0"),
    ("pydantic", "pydantic>=2.5.0"),
    ("yaml", "pyyaml>=6.0.0"),
    ("openai", "openai>=1.10.0"),
    ("google.generativeai", "google-generativeai>=0.3.0"),
    ("pandas", "pandas>=2.0.0"),
]


def _run(cmd: list[str]) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


def _missing_packages() -> list[str]:
    missing: list[str] = []
    for import_name, spec in RUNTIME_PACKAGES:
        root = import_name.split(".", 1)[0]
        if importlib.util.find_spec(root) is None:
            missing.append(spec)
    return missing


def _install_packages(specs: list[str]) -> None:
    """Install into the *current* kernel env. Prefer uv (this repo); fall back to pip."""
    if not specs:
        return
    if shutil.which("uv"):
        _run(["uv", "pip", "install", "--python", sys.executable, *specs])
        return
    # Some envs (uv venvs) ship without pip — bootstrap if needed.
    try:
        import pip  # noqa: F401
    except ModuleNotFoundError:
        _run([sys.executable, "-m", "ensurepip", "--upgrade"])
    _run([sys.executable, "-m", "pip", "install", "-q", *specs])


def _find_repo_root(start: str) -> str | None:
    """Walk up from `start` looking for the project (services/ + config/topics.py)."""
    path = os.path.abspath(start)
    while True:
        has_services = os.path.isdir(os.path.join(path, "services"))
        has_topics = os.path.isfile(os.path.join(path, "config", "topics.py"))
        if has_services and has_topics:
            return path
        parent = os.path.dirname(path)
        if parent == path:
            return None
        path = parent


repo_root = _find_repo_root(os.getcwd())
if repo_root is None:
    if not os.path.isdir(REPO_DIR):
        _run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL])
    repo_root = os.path.abspath(REPO_DIR)

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

missing = _missing_packages()
if missing:
    print("Installing missing packages into this kernel:", ", ".join(missing))
    _install_packages(missing)
else:
    print("Runtime packages already present.")

# Verify pandas is available for the Results cell.
import pandas as pd  # noqa: E402

print("Repo root:", repo_root)
print("pandas:", pd.__version__)
print("Setup complete.")


Runtime packages already present.
Repo root: /Users/bakulkumarkakadiya/dev/github/portfolio-top-movers
pandas: 3.0.3
Setup complete.


## 2. API keys

Needs a **Bigdata** API key (company resolution, prices, news) and **one** LLM key (OpenAI *or* Gemini) for the summaries.

Keys are read from the environment if already set; otherwise you'll be prompted (hidden input). Nothing is written to disk.

In [22]:
import getpass
import os


def _ensure_env(name: str, prompt: str) -> bool:
    """Populate an env var from a hidden prompt if it isn't already set."""
    if not os.getenv(name):
        try:
            value = getpass.getpass(prompt).strip()
        except Exception:
            value = ""
        if value:
            os.environ[name] = value
    return bool(os.getenv(name))


_ensure_env("BIGDATA_API_KEY", "Bigdata API key: ")

# Need exactly one LLM provider. Prefer whatever is already in the environment.
if not (os.getenv("OPENAI_API_KEY") or os.getenv("GEMINI_API_KEY")):
    _ensure_env("OPENAI_API_KEY", "OpenAI API key (leave blank to use Gemini instead): ")
    if not os.getenv("OPENAI_API_KEY"):
        _ensure_env("GEMINI_API_KEY", "Gemini API key: ")

_llm = "OpenAI" if os.getenv("OPENAI_API_KEY") else ("Gemini" if os.getenv("GEMINI_API_KEY") else None)
print("Bigdata key:", "set" if os.getenv("BIGDATA_API_KEY") else "MISSING")
print("LLM provider:", _llm or "MISSING")

if not os.getenv("BIGDATA_API_KEY"):
    raise RuntimeError("BIGDATA_API_KEY is required.")
if _llm is None:
    raise RuntimeError("Set OPENAI_API_KEY or GEMINI_API_KEY for LLM summaries.")

BIGDATA_API_KEY = os.environ["BIGDATA_API_KEY"]
BIGDATA_BASE_URL = "https://api.bigdata.com/v1"

Bigdata key: set
LLM provider: OpenAI


## 3. Config panel  (edit me)

This is the only cell most users need to change. Re-run it (and the cells below) after editing.

- **`TICKERS`** — the universe to rank. More names = more meaningful gainers/decliners.
- **`TIME_PERIOD_DAYS`** — lookback for *both* the news window and the price-change window used for ranking. `1`=today, `5`≈week, `30`≈month, `90`≈quarter, `180`≈half-year, `365`≈year.
- **`TOP_N`** — how many gainers and how many decliners to report.
- **`INCLUDE_TOPICS`** — named topics to include (see the catalog printed in the next cell). Set to `None` to use all.
- **`CUSTOM_TOPICS`** — your own topics; use `{company}` as a placeholder.
- **`MAX_SUMMARY_BULLETS`** — hard cap on bullets per stock (most important first).

Duplicate tickers in `TICKERS` are automatically removed (order preserved).


In [23]:
# ------------------------------------------------------------------ #
#  CONFIG PANEL  -- edit these values                                #
# ------------------------------------------------------------------ #

# Universe to rank. Accepts "AAPL" or "XNAS:AAPL"; exchange prefixes are stripped.
# Duplicates are removed automatically in the pipeline (order preserved).
TICKERS = [
    "AAPL", "AMZN", "TSLA", "NVDA", "AVGO", "INTC",
    "MSFT", "GOOGL", "META", "TSM", "NFLX",
]

# Lookback (days) for BOTH the news search window AND the price-change ranking window.
TIME_PERIOD_DAYS = 1

# Number of top gainers and top decliners to report.
TOP_N = 2

# Named topics to include. Names come from the catalog printed in the next cell.
# Set to None to include every catalog topic.
INCLUDE_TOPICS = ["Earnings", "Analyst", "Products", "Supply Chain", "M&A"]

# Your own topics. Use {company} as a placeholder for the company name.
CUSTOM_TOPICS = [
    {"topic_name": "Product News",
     "topic_text": "{company} new product launch feature release or major update announcement"},
]

# Hard cap on summary bullets per stock (ranked most-important first).
MAX_SUMMARY_BULLETS = 3

print(f"{len(TICKERS)} tickers | {TIME_PERIOD_DAYS}-day window | top {TOP_N} each way | "
      f"max {MAX_SUMMARY_BULLETS} bullets/stock")


11 tickers | 1-day window | top 2 each way | max 3 bullets/stock


In [24]:
print("Tickers:", TICKERS)
print("Time period:", TIME_PERIOD_DAYS)
print("Top N:", TOP_N)
print("Include topics:", INCLUDE_TOPICS)
print("Custom topics:", CUSTOM_TOPICS)
print("Max summary bullets:", MAX_SUMMARY_BULLETS)


Tickers: ['AAPL', 'AMZN', 'TSLA', 'NVDA', 'AVGO', 'INTC', 'MSFT', 'GOOGL', 'META', 'TSM', 'NFLX']
Time period: 1
Top N: 2
Include topics: ['Earnings', 'Analyst', 'Products', 'Supply Chain', 'M&A']
Custom topics: [{'topic_name': 'Product News', 'topic_text': '{company} new product launch feature release or major update announcement'}]
Max summary bullets: 3


## 4. Build topics

Shows the available named topics and assembles the final search topics from your `INCLUDE_TOPICS` + `CUSTOM_TOPICS`.



In [25]:
from collections import OrderedDict

from config.topics import MOVERS_TOPICS, STANDARD_TOPICS

# Catalog: group every template by its topic name (desk-style + movers sets combined).
TOPIC_CATALOG: "OrderedDict[str, list[str]]" = OrderedDict()
for _t in STANDARD_TOPICS + MOVERS_TOPICS:
    TOPIC_CATALOG.setdefault(_t["topic_name"], []).append(_t["topic_text"])

print("Available named topics (query templates each):")
for _name, _texts in TOPIC_CATALOG.items():
    print(f"  - {_name}  ({len(_texts)})")


def build_topics(include_topics, custom_topics):
    """Assemble {topic_name, topic_text} templates from selected names + custom topics."""
    names = list(TOPIC_CATALOG.keys()) if include_topics is None else list(include_topics)

    known = {n.lower(): n for n in TOPIC_CATALOG}
    topics, missing = [], []
    for name in names:
        canonical = known.get(name.lower())
        if canonical is None:
            missing.append(name)
            continue
        topics.extend({"topic_name": canonical, "topic_text": txt} for txt in TOPIC_CATALOG[canonical])

    for ct in custom_topics or []:
        topics.append({"topic_name": ct["topic_name"], "topic_text": ct["topic_text"]})

    if missing:
        print(f"\n[warning] not in catalog (ignored): {missing}. "
              f"Add them via CUSTOM_TOPICS or fix the spelling.")
    return topics


SEARCH_TOPICS = build_topics(INCLUDE_TOPICS, CUSTOM_TOPICS)
_selected_names = sorted({t["topic_name"] for t in SEARCH_TOPICS})
print(f"\nSelected {len(_selected_names)} topic groups -> {len(SEARCH_TOPICS)} query templates:")
print("  " + ", ".join(_selected_names))

Available named topics (query templates each):
  - Financial Metrics  (3)
  - M&A  (5)
  - Leadership  (3)
  - Competition  (3)
  - Products  (1)
  - Supply Chain  (3)
  - Costs  (1)
  - Regulatory  (4)
  - Industry  (3)
  - Financing  (2)
  - Earnings  (2)
  - Analyst  (2)
  - Corporate Action  (2)
  - Business  (2)
  - Market  (2)

Selected 6 topic groups -> 14 query templates:
  Analyst, Earnings, M&A, Product News, Products, Supply Chain


## 5. Helpers

Two small building blocks used by the pipeline:

- **Windowed price change** — maps `TIME_PERIOD_DAYS` to the nearest window (`1D/5D/1M/3M/6M/1Y`) returned by `/price/changes/query` and ranks on it. This is why changing the period now changes the movers (the app only ever read `1D`).
- **Top-N bullet summarizer** — turns per-topic briefs into at most `MAX_SUMMARY_BULLETS`, ranked by materiality.


In [26]:
import requests

# Windows available from /price/changes/query, in ascending duration (days, key).
_CHANGE_WINDOWS = [(1, "1D"), (5, "5D"), (30, "1M"), (90, "3M"), (180, "6M"), (365, "1Y")]


def days_to_change_window(days: int) -> str:
    """Map a lookback in days to the nearest supported price-change window key."""
    return min(_CHANGE_WINDOWS, key=lambda w: abs(w[0] - days))[1]


def change_for_window(entity_id: str, days: int, api_key: str) -> tuple[float | None, str]:
    """Return (percent_change, window_key) for the window nearest to `days`."""
    window = days_to_change_window(days)
    if not entity_id:
        return None, window
    try:
        resp = requests.post(
            f"{BIGDATA_BASE_URL}/price/changes/query",
            headers={"X-API-KEY": api_key, "Content-Type": "application/json"},
            json={"identifier": {"type": "rp_entity_id", "value": entity_id}},
            timeout=15,
        )
        resp.raise_for_status()
        results = resp.json().get("results", [])
        if results:
            return results[0].get(window), window
    except requests.RequestException as exc:
        print(f"[warning] price change failed for {entity_id}: {exc}")
    return None, window


RANKING_WINDOW = days_to_change_window(TIME_PERIOD_DAYS)
print(f"Ranking movers on the {RANKING_WINDOW} price-change window "
      f"(nearest to {TIME_PERIOD_DAYS} day(s)).")

Ranking movers on the 1D price-change window (nearest to 1 day(s)).


In [27]:
from pydantic import BaseModel

from services.report_service import ReportService, TopicBrief

# One ReportService for the whole notebook; reuse its LLM for the summarizer.
report_service = ReportService()
llm_service = report_service.llm_service
print(f"LLM ready: provider={llm_service.provider_name}, model={llm_service.model}")


class RankedBullet(BaseModel):
    """A single prioritized summary bullet."""
    rank: int
    topic_name: str
    bullet: str


async def summarize_top_bullets(
    briefs: list[TopicBrief],
    company_name: str,
    max_bullets: int = 3,
) -> list[RankedBullet]:
    """Select and rewrite the `max_bullets` most material briefs, ranked (1 = top)."""
    if not briefs:
        return []

    candidates = "\n".join(
        f"- [{b.topic_name}] {b.bullet_point}" for b in briefs
    )
    prompt = (
        f"You are an equity analyst. Below are candidate one-line briefs about "
        f"{company_name}, one per topic.\n\n"
        f"Candidate briefs:\n{candidates}\n\n"
        f"Select the {max_bullets} MOST important, market-moving items. Rank them by "
        f"materiality (rank 1 = most important). Rewrite each as a single concise, "
        f"information-dense sentence. Use ONLY the information in the candidates; do not "
        f"speculate or add outside facts. Keep the original topic_name for each. "
        f"Return at most {max_bullets} bullets."
    )

    bullets = await llm_service.generate_content_list(
        prompt=prompt,
        response_schema=RankedBullet,
    )
    bullets = sorted(bullets, key=lambda b: b.rank)[:max_bullets]
    return bullets

LLM ready: provider=openai, model=gpt-5-mini


## 6. Run pipeline

Resolves entities, ranks movers on the windowed change (deduping tickers so each symbol appears once), searches news over the same window with your topics, then builds the capped summaries. Reuses the tested helpers in [`services/movers_workflow.py`](../services/movers_workflow.py).


In [28]:
import asyncio

from services.movers_workflow import (
    MoverData,
    fetch_entity_ids_batch,
    fetch_news_for_mover,
    find_top_movers,
    normalize_ticker,
)
from services.price_service import get_latest_price
from services.topic_search_service import TopicSearchService


def _unique_tickers(raw: list[str]) -> list[str]:
    """Normalize and dedupe tickers, preserving order."""
    return list(dict.fromkeys(normalize_ticker(t) for t in raw if str(t).strip()))


def _dedupe_movers(movers: list[MoverData]) -> list[MoverData]:
    """Keep the first occurrence of each ticker."""
    seen: set[str] = set()
    unique: list[MoverData] = []
    for m in movers:
        if m.ticker in seen:
            continue
        seen.add(m.ticker)
        unique.append(m)
    return unique


def _exclusive_rank(movers: list[MoverData], top_n: int) -> dict[str, list[MoverData]]:
    """Top gainers / decliners with each ticker appearing at most once overall."""
    movers = _dedupe_movers(movers)
    ranked = find_top_movers(movers, top_n)
    gainers = _dedupe_movers(ranked["gainers"])[:top_n]
    taken = {m.ticker for m in gainers}

    # True decliners first (most negative), never overlapping gainers.
    candidates = sorted(
        [m for m in movers if m.price_change_pct is not None and m.ticker not in taken],
        key=lambda m: m.price_change_pct or 0,
    )
    decliners = [m for m in candidates if (m.price_change_pct or 0) < 0][:top_n]
    if len(decliners) < top_n:
        # Backfill with remaining lowest movers not already selected.
        for m in candidates:
            if m.ticker in {d.ticker for d in decliners}:
                continue
            decliners.append(m)
            if len(decliners) >= top_n:
                break
    return {"gainers": gainers, "decliners": decliners[:top_n]}


async def _process_mover(mover: MoverData, service: TopicSearchService) -> dict:
    """News -> briefs -> top-N bullets for one stock."""
    mover = await fetch_news_for_mover(
        mover, service, days=TIME_PERIOD_DAYS, custom_topics=SEARCH_TOPICS,
    )
    topic_results = (mover.news_data or {}).get("topic_results", []) if mover.news_data else []

    news_response = {
        "ticker": mover.ticker,
        "company_name": mover.company_name,
        "topic_results": topic_results,
    }
    briefs = await report_service.generate_topic_briefs(news_response) if topic_results else []
    bullets = await summarize_top_bullets(briefs, mover.company_name, MAX_SUMMARY_BULLETS)

    return {
        "mover": mover,
        "bullets": bullets,
        "articles": len(topic_results),
    }


async def run_pipeline() -> dict:
    service = TopicSearchService(api_key=BIGDATA_API_KEY, base_url=BIGDATA_BASE_URL)
    try:
        tickers = _unique_tickers(TICKERS)
        if len(tickers) < len([t for t in TICKERS if str(t).strip()]):
            print(f"Deduped tickers: {len(TICKERS)} -> {len(tickers)} ({', '.join(tickers)})")

        print("Resolving entities...")
        entity_data = await fetch_entity_ids_batch(tickers, service)

        print(f"Fetching {RANKING_WINDOW} price changes...")
        movers: list[MoverData] = []
        for tk in tickers:
            info = entity_data.get(tk, {})
            entity_id = info.get("entity_id")
            pct, _ = change_for_window(entity_id, TIME_PERIOD_DAYS, BIGDATA_API_KEY)
            price_info = get_latest_price(entity_id, tk, BIGDATA_API_KEY) if entity_id else None
            movers.append(MoverData(
                ticker=tk,
                company_name=info.get("company_name", tk),
                entity_id=entity_id,
                current_price=(price_info or {}).get("price"),
                price_change_pct=pct,
                currency=(price_info or {}).get("currency", "USD"),
            ))

        ranked = _exclusive_rank(movers, TOP_N)
        print(
            f"Ranked {len(ranked['gainers'])} gainers / {len(ranked['decliners'])} decliners. "
            f"Building news + summaries..."
        )

        results = {"window": RANKING_WINDOW, "gainers": [], "decliners": []}
        for group in ("gainers", "decliners"):
            results[group] = list(
                await asyncio.gather(*[_process_mover(m, service) for m in ranked[group]])
            )
        return results
    finally:
        await service.close()


RESULTS = await run_pipeline()
print("Pipeline complete.")


Resolving entities...
Fetching 1D price changes...
Ranked 2 gainers / 2 decliners. Building news + summaries...
Pipeline complete.


## 7. Results

A ranking table, then per-stock: the capped top-N bullets.


In [29]:
import pandas as pd
from IPython.display import Markdown, display

_window = RESULTS["window"]


def _fmt_pct(x):
    return f"{x:+.2f}%" if isinstance(x, (int, float)) else "N/A"


def _fmt_price(x):
    return f"{x:,.2f}" if isinstance(x, (int, float)) else "N/A"


# --- Ranking table -------------------------------------------------- #
_rows = []
_seen_keys: set[tuple[str, str]] = set()
for _group, _label in (("gainers", "Gainer"), ("decliners", "Decliner")):
    for _r in RESULTS[_group]:
        _m = _r["mover"]
        _key = (_label, _m.ticker)
        if _key in _seen_keys:
            continue
        _seen_keys.add(_key)
        _rows.append({
            "Type": _label,
            "Ticker": _m.ticker,
            "Company": _m.company_name,
            "Price": _fmt_price(_m.current_price),
            f"{_window} Change": _fmt_pct(_m.price_change_pct),
            "Articles": _r["articles"],
            "Bullets": len(_r["bullets"]),
        })

display(Markdown(f"### Movers ranked on the {_window} price-change window"))
display(pd.DataFrame(_rows))

# --- Per-stock detail ----------------------------------------------- #
_shown: set[str] = set()
for _group, _heading in (("gainers", "Top Gainers"), ("decliners", "Top Decliners")):
    if not RESULTS[_group]:
        continue
    display(Markdown(f"## {_heading}"))
    for _r in RESULTS[_group]:
        _m = _r["mover"]
        _key = f"{_group}:{_m.ticker}"
        if _key in _shown:
            continue
        _shown.add(_key)
        display(Markdown(
            f"### {_m.company_name} ({_m.ticker}) — {_fmt_pct(_m.price_change_pct)} "
            f"over {_window}  ·  {_r['articles']} articles"
        ))

        if _r["bullets"]:
            _lines = "\n".join(
                f"{_b.rank}. **[{_b.topic_name}]** {_b.bullet}" for _b in _r["bullets"]
            )
        else:
            _lines = "_No significant news found in the selected window / topics._"
        display(Markdown(
            f"**Summary (max {MAX_SUMMARY_BULLETS} bullets):**\n\n{_lines}"
        ))


### Movers ranked on the 1D price-change window

,Type,Ticker,Company,Price,1D Change,Articles,Bullets
0,Gainer,AAPL,Apple Inc.,327.54,+4.01%,111,3
1,Gainer,GOOGL,Alphabet Inc.,370.85,+3.17%,35,3
2,Decliner,INTC,Intel Corp.,102.93,-4.43%,51,3
3,Decliner,TSLA,Tesla Inc.,394.46,-0.46%,49,3


## Top Gainers

### Apple Inc. (AAPL) — +4.01% over 1D  ·  111 articles

**Summary (max 3 bullets):**

1. **[Supply Chain]** Apple Inc. Memory-driven DRAM shortage is materially pressuring margins — memory costs already hit the March quarter and are expected to rise significantly in the June quarter and beyond as AI/data centers consume ~70% of supply, forcing price hikes and shipment constraints.
2. **[Earnings]** Apple Inc. guided June-quarter gross margin to 47.5–48.5% despite surging memory costs, and Morgan Stanley estimates product price hikes will add 2–4% to Q3 EPS and roughly 1% to FY27 EPS, supporting near-term profit resilience.
3. **[M&A]** Apple Inc. is actively courting semiconductor and AI‑chip startups to accelerate delayed 'Baltra' AI server‑chip development after relying on Nvidia/Google Cloud, and the CFO signalled abandoning 'net cash neutral', enabling larger strategic M&A.

...

## Top Decliners

### Intel Corp. (INTC) — -4.43% over 1D  ·  51 articles

**Summary (max 3 bullets):**

1. **[Earnings]** Intel Corp. reports Q2 results July 23; the street expects $0.19 EPS on ~ $14.4B revenue, management guidance $13.8–14.8B, and prediction markets show ~67–75% odds of beating key Foundry/Data Center thresholds.
2. **[M&A]** Intel Corp. struck a strategic Terafab partnership with SpaceX, xAI and Tesla to produce up to one terawatt/year of AI compute using Intel’s 14A technology (still in final development), providing a marquee high‑volume foundry customer.
3. **[Supply Chain]** Intel Corp. began high‑volume production using ASML High‑NA EUV on selected 18A layers for Panther Lake, with yields matching NXE and 18A capacity expanding to ~30,000 wafers/month across Phoenix and Hillsboro.

### Tesla Inc. (TSLA) — -0.46% over 1D  ·  49 articles

**Summary (max 3 bullets):**

1. **[Earnings]** Tesla Inc. set to report Q2 results on July 22; ~480,100 Q2 deliveries (up ~25% YoY) support expected revenue and EPS growth, but management raised 2026 capex to $25B, increasing the risk of negative free cash flow.
2. **[Product News]** Tesla Inc. completed tape-out of its AI5 chip on a 2‑nanometer node and is preparing mass production at Samsung's Taylor, Texas fab, targeting autonomous vehicles and Tesla data‑center workloads.
3. **[Supply Chain]** Tesla Inc. began volume production of the Tesla Semi in 2026, launched PTI pilot evaluations and Megacharger deployment, but Musk cautioned initial output will ramp slowly with acceleration expected into 2027.